# Must-survive budget

Constructs a truncation budget where an old required user turn is dropped, then validates the minimized survival counterexample.


The code cell below builds a minimal in-memory artifact, runs the real PromptABI checker, and asserts the proof obligation.


In [ ]:
from promptabi import ArtifactKind, ArtifactLocation, FrameworkTruncationConfigArtifact
from promptabi import PromptSegment, PromptSegmentArtifact, TruncationStrategy, VerificationConfig
from promptabi import analyze_token_budget, prove_must_survive_budget
from promptabi.loaders import ArtifactLoader
from promptabi.proof_sketches import ProofOutcome

segments = PromptSegmentArtifact(
    kind=ArtifactKind.PROMPT_SEGMENT,
    name="conversation",
    location=ArtifactLocation(uri="memory://segments"),
    segments=(
        PromptSegment("system", role="system", required=True, token_count=18),
        PromptSegment("old-user", role="user", required=True, token_count=26),
        PromptSegment("latest-user", role="user", required=True, token_count=20),
    ),
)
budget = FrameworkTruncationConfigArtifact(
    kind=ArtifactKind.FRAMEWORK_TRUNCATION_CONFIG,
    name="langchain-budget",
    location=ArtifactLocation(uri="memory://budget"),
    framework="langchain",
    strategy=TruncationStrategy.OLDEST_MESSAGE,
    max_context_tokens=50,
)
report = analyze_token_budget(
    VerificationConfig(name="budget"),
    (ArtifactLoader().load(segments), ArtifactLoader().load(budget)),
)
sketch = prove_must_survive_budget(report.must_survive_proof)

assert sketch.outcome is ProofOutcome.COUNTEREXAMPLE
assert sketch.passed
assert dict(sketch.evidence)["dropped_required"] == "old-user"
sketch.to_dict()
